In [70]:
import pickle

#loading dataset(loading the full Label array so that we can create window labels)
with open("../data/WESAD/S2/S2.pkl", "rb") as file:
    data = pickle.load(file, encoding='latin1')

labels = data['label']

C:\Users\PC\AppData\Local\Temp\ipykernel_8792\3448557206.py:5: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data = pickle.load(file, encoding='latin1')


In [71]:
#loading all wrist signals to notebook
wrist = data['signal']['wrist']

eda = wrist['EDA'].flatten()
bvp = wrist['BVP'].flatten()
acc = wrist['ACC']
temp = wrist['TEMP'].flatten()


#checkin shapes
print("EDA:", eda.shape)
print("BVP:", bvp.shape)
print("ACC:", acc.shape)
print("TEMP:", temp.shape)


EDA: (24316,)
BVP: (389056,)
ACC: (194528, 3)
TEMP: (24316,)


In [72]:
#load windows that was defined in notebook1 so that the "apply to all windows" cell recognizes windows
import numpy as np

windows = np.load("../data/processed/eda_windows_s2.npy")

print("Loaded windows shape:", windows.shape)


Loaded windows shape: (101, 240)


In [73]:
import numpy as np
from scipy.signal import find_peaks

def extract_eda_features(window, fs=4):
    features = {}
    
    # Basic statistics
    features["mean"] = np.mean(window)
    features["std"] = np.std(window)
    features["min"] = np.min(window)
    features["max"] = np.max(window)
    
    # Linear slope (trend)
    x = np.arange(len(window))
    slope = np.polyfit(x, window, 1)[0]
    features["slope"] = slope
    
    # Peak detection (simple SCR estimation)
    peaks, _ = find_peaks(window, distance=fs)  
    features["num_peaks"] = len(peaks)
    
    return features


In [74]:
#apply to all windows
feature_list = []

for window in windows:
    features = extract_eda_features(window)
    feature_list.append(features)

import pandas as pd

eda_feature_df = pd.DataFrame(feature_list)

eda_feature_df.head()


,mean,std,min,max,slope,num_peaks
0,1.232191,0.101280,0.934426,1.616094,0.000471,24
1,1.096766,0.063136,0.930582,1.239383,0.000032,19
2,1.030441,0.118612,0.851140,1.405956,0.000836,17
3,1.002630,0.083628,0.843452,1.188130,-0.000552,15
4,1.239722,0.185494,0.917769,1.717419,0.001943,26


In [75]:
fs_eda = 4
fs_label = 700
window_seconds = 60

eda_window_size = fs_eda * window_seconds
label_window_size = fs_label * window_seconds

window_labels = []

for i in range(len(windows)):
    
    label_start = i * label_window_size
    label_end = label_start + label_window_size
    
    label_segment = labels[label_start:label_end]
    
    # Keep only valid labels (1,2,3)
    valid = label_segment[np.isin(label_segment, [1,2,3])]
    
    if len(valid) == 0:
        window_labels.append(-1)
    else:
        majority_label = np.bincount(valid).argmax()
        window_labels.append(majority_label)

window_labels = np.array(window_labels)


In [76]:
#remove invalid windows
valid_mask = window_labels != -1

feature_df = eda_feature_df[valid_mask]
window_labels = window_labels[valid_mask]

print("Final dataset shape:", feature_df.shape)


Final dataset shape: (39, 6)


In [77]:
unique, counts = np.unique(window_labels, return_counts=True)

for u, c in zip(unique, counts):
    print(f"Label {u}: {c} windows")


Label 1: 20 windows
Label 2: 12 windows
Label 3: 7 windows


In [78]:
#BVP

In [79]:
#Create BVP windows in this notebook

from scipy.interpolate import interp1d
from scipy.signal import welch
from scipy.signal import find_peaks
def create_windows(signal, fs, window_seconds=60):
    window_size = fs * window_seconds
    windows = []
    for start in range(0, len(signal) - window_size + 1, window_size):
        end = start + window_size
        windows.append(signal[start:end])
    return np.array(windows)

bvp_windows = create_windows(bvp, 64, window_seconds=60)
print("BVP windows shape:", bvp_windows.shape)


BVP windows shape: (101, 3840)


In [80]:
#LF/HF Calculation
#The goal is to compute frequency domain HRV from inter-beat intervals(IBIs)
#LF (Low Frequency): 0.04–0.15 Hz → mostly sympathetic + some parasympathetic
#HF (High Frequency): 0.15–0.4 Hz → parasympathetic only
#LF/HF ratio: Stress ↑ sympathetic → LF/HF ↑

#first step is computing interpolated IBI signal
#HRV frequency analysus requires evenly spaced samples but IBIs are irregular
from scipy.interpolate import interp1d
from scipy.signal import welch
from scipy.signal import find_peaks

def compute_lfhf(ibi, fs_interp=4):
    # ibi in ms → convert to seconds
    t = np.cumsum(ibi) / 1000
    # remove first element to align
    t = t[:-1]
    ibi = ibi[1:]
    
    # Interpolation
    f = interp1d(t, ibi, kind='cubic', fill_value="extrapolate")
    t_interp = np.arange(t[0], t[-1], 1/fs_interp)
    ibi_interp = f(t_interp)
    
    # Compute power spectral density
    f_psd, p_psd = welch(ibi_interp, fs=fs_interp, nperseg=min(256, len(ibi_interp)))
    
    # LF band 0.04–0.15 Hz
    lf = np.trapezoid(p_psd[(f_psd >= 0.04) & (f_psd <= 0.15)], f_psd[(f_psd >= 0.04) & (f_psd <= 0.15)])
    # HF band 0.15–0.4 Hz
    hf = np.trapezoid(p_psd[(f_psd >= 0.15) & (f_psd <= 0.4)], f_psd[(f_psd >= 0.15) & (f_psd <= 0.4)])
    
    lf_hf = lf / hf if hf != 0 else 0
    return lf, hf, lf_hf
#Integrate into extract_bvp_features down below

In [81]:

#Detect peaks in BVP
#I need heartbeat times to compute HRV features.
from scipy.signal import find_peaks

def extract_bvp_features(window, fs=64):
    features = {}
    
    # Detect peaks (heartbeat locations)
    peaks, _ = find_peaks(window, distance=fs*0.5)  # min 0.5 s between beats
    ibi = np.diff(peaks) / fs * 1000  # convert to ms
    
    # Heart rate
    hr = 60 / (ibi / 1000)  # bpm
    features['HR_mean'] = np.mean(hr)
    features['HR_std'] = np.std(hr)
    
    # HRV metrics time-domain
    features['RMSSD'] = np.sqrt(np.mean(np.diff(ibi)**2))
    features['pNN50'] = np.sum(np.abs(np.diff(ibi)) > 50) / len(ibi) * 100
    
    # LF/HF  simplified version
    # Here I can later do FFT for LF/HF
    # For now I will leave placeholders
    #features['LF'] = 0
   # features['HF'] = 0
    #features['LF_HF'] = 0

    
    # Frequency-domain HRV
    if len(ibi) > 4:  # need enough beats for PSD
        lf, hf, lf_hf = compute_lfhf(ibi)
        features['LF'] = lf
        features['HF'] = hf
        features['LF_HF'] = lf_hf
    else:
        features['LF'] = 0
        features['HF'] = 0
        features['LF_HF'] = 0
    return features


In [82]:
#apply to all windows
bvp_features_list = [extract_bvp_features(win) for win in bvp_windows]
bvp_feature_df = pd.DataFrame(bvp_features_list)
print("BVP feature shape:", bvp_feature_df.shape)
bvp_feature_df.head()



BVP feature shape: (101, 7)


,HR_mean,HR_std,RMSSD,pNN50,LF,HF,LF_HF
0,76.740279,13.456074,184.785765,69.863014,2993.344520,9356.864221,0.319909
1,78.591827,12.026328,149.728487,40.789474,2204.713693,1787.666600,1.233291
2,75.847404,11.107834,137.838644,41.891892,3124.351380,3397.273216,0.919664
3,77.368731,18.150894,268.336599,68.493151,7326.274648,23272.857252,0.314799
4,82.254983,20.521985,313.498405,69.736842,3683.585002,31359.376449,0.117464


In [83]:
#ACC

In [84]:
#create acc windows
def create_windows(data, window_size, step_size):
    windows = []
    for start in range(0, len(data) - window_size, step_size):
        windows.append(data[start:start + window_size])
    return windows

#definitions for acc xtics
acc_fs = 32
window_seconds = 60

acc_window_size = acc_fs * window_seconds
acc_step_size = acc_window_size  # no overlap for now

acc_windows = create_windows(acc, acc_window_size, acc_step_size)

print("Number of ACC windows:", len(acc_windows))


Number of ACC windows: 101


In [85]:
#test to make sure windowing is consistent they should not be different, must be equal
print(len(bvp_windows))
print(len(acc_windows))


101
101


In [86]:
#acc feature function
def extract_acc_features(window):
    features = {}

    # Split axes
    x = window[:, 0]
    y = window[:, 1]
    z = window[:, 2]

    # Per-axis statistics
    features['ACC_x_mean'] = np.mean(x)
    features['ACC_x_std'] = np.std(x)

    features['ACC_y_mean'] = np.mean(y)
    features['ACC_y_std'] = np.std(y)

    features['ACC_z_mean'] = np.mean(z)
    features['ACC_z_std'] = np.std(z)

    # Magnitude
    magnitude = np.sqrt(x**2 + y**2 + z**2)

    features['ACC_mag_mean'] = np.mean(magnitude)
    features['ACC_mag_std'] = np.std(magnitude)

    return features


In [87]:
print(acc.shape)


(194528, 3)


In [88]:
#apply to all windows
acc_features_list = [extract_acc_features(win) for win in acc_windows]
acc_feature_df = pd.DataFrame(acc_features_list)

print("ACC feature shape:", acc_feature_df.shape)
acc_feature_df.head()


ACC feature shape: (101, 8)


,ACC_x_mean,ACC_x_std,ACC_y_mean,ACC_y_std,ACC_z_mean,ACC_z_std,ACC_mag_mean,ACC_mag_std
0,27.756771,18.018529,-25.642187,40.006518,18.956771,18.217943,63.359597,5.473853
1,43.852604,7.112463,-2.172917,21.156050,36.278125,15.834487,63.016435,4.658349
2,44.015625,10.681651,-9.135417,12.764510,40.478646,12.622548,63.453179,8.330967
3,37.696354,14.638384,3.318229,26.511599,39.450521,14.765551,63.633747,8.618834
4,54.628125,13.204807,-1.247917,17.412400,5.553125,23.947605,63.536801,5.545078


In [89]:
#TEMP


In [90]:
#creating Temp Windows
temp_fs = 4
window_seconds = 60
temp_window_size = temp_fs * window_seconds

temp_windows = create_windows(temp, temp_window_size, temp_window_size)
print("Number of TEMP windows:", len(temp_windows))#to confirm consistency with other signals


Number of TEMP windows: 101


In [91]:
#TEMP feature extraction
def extract_temp_features(window):
    features = {}
    
    features['TEMP_mean'] = np.mean(window)
    features['TEMP_std'] = np.std(window)
    features['TEMP_min'] = np.min(window)
    features['TEMP_max'] = np.max(window)
    features['TEMP_slope'] = (window[-1] - window[0]) / len(window)
    
    return features


In [92]:
#actual extract temp feattures(apply to all windows)
temp_features_list = [extract_temp_features(win) for win in temp_windows]
temp_feature_df = pd.DataFrame(temp_features_list)

print("TEMP feature shape:", temp_feature_df.shape)
temp_feature_df.head()


TEMP feature shape: (101, 5)


,TEMP_mean,TEMP_std,TEMP_min,TEMP_max,TEMP_slope
0,35.460833,0.043023,35.41,35.55,0.000500
1,35.519333,0.028511,35.49,35.59,0.000000
2,35.670000,0.094921,35.50,35.84,0.001167
3,35.809167,0.017540,35.77,35.87,-0.000083
4,35.790833,0.020999,35.73,35.84,0.000167


In [93]:
#Now, Merging all features into one dataset for my model

In [94]:
#Concatenating all features to give me one big feature matrix
# Combine along columns
features_df = pd.concat(
    [bvp_feature_df, eda_feature_df, acc_feature_df, temp_feature_df],
    axis=1
)

print("Combined feature matrix shape:", features_df.shape)
features_df.head()


Combined feature matrix shape: (101, 26)


,HR_mean,HR_std,RMSSD,pNN50,LF,HF,LF_HF,mean,std,min,...,ACC_y_std,ACC_z_mean,ACC_z_std,ACC_mag_mean,ACC_mag_std,TEMP_mean,TEMP_std,TEMP_min,TEMP_max,TEMP_slope
0,76.740279,13.456074,184.785765,69.863014,2993.344520,9356.864221,0.319909,1.232191,0.101280,0.934426,...,40.006518,18.956771,18.217943,63.359597,5.473853,35.460833,0.043023,35.41,35.55,0.000500
1,78.591827,12.026328,149.728487,40.789474,2204.713693,1787.666600,1.233291,1.096766,0.063136,0.930582,...,21.156050,36.278125,15.834487,63.016435,4.658349,35.519333,0.028511,35.49,35.59,0.000000
2,75.847404,11.107834,137.838644,41.891892,3124.351380,3397.273216,0.919664,1.030441,0.118612,0.851140,...,12.764510,40.478646,12.622548,63.453179,8.330967,35.670000,0.094921,35.50,35.84,0.001167
3,77.368731,18.150894,268.336599,68.493151,7326.274648,23272.857252,0.314799,1.002630,0.083628,0.843452,...,26.511599,39.450521,14.765551,63.633747,8.618834,35.809167,0.017540,35.77,35.87,-0.000083
4,82.254983,20.521985,313.498405,69.736842,3683.585002,31359.376449,0.117464,1.239722,0.185494,0.917769,...,17.412400,5.553125,23.947605,63.536801,5.545078,35.790833,0.020999,35.73,35.84,0.000167


In [96]:
X = features_df.values  # shape (101, 25)
y = np.array(window_labels)  # shape (101,)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 5 labels:", y[:5])


X shape: (101, 26)
y shape: (39,)
First 5 labels: [1 1 1 1 1]


In [97]:
#Saving these features  to avoid future re extraction
features_df.to_csv("Integrisense_features.csv", index=False)
np.save("X.npy", X)
np.save("y.npy", y)
